In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy
import math

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print("distribute_microorganism_initial: ", distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    print(print("distribute_public_metabolite_initial: ", distribute_public_metabolism_list['glc__D_e'][0::]))
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
            
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
            
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
        
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0]
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        pass
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
            
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
            
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                        
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])
                                
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death        
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                    
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                        
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
            
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
        
        print("distribute_public_metabolite: ", distribute_public_metabolism_list['glc__D_e'][0::])
        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
        
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
       
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    print("distribute_microorganism: ", distribute_micro_list_t)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

distribute_microorganism_initial:  {0: array([37, 20, 33,  5, 29, 13, 22, 26, 15, 29, 39, 28, 12, 27, 14, 12, 32,
       16,  6]), 1: array([37, 10, 13,  2, 26,  8, 31, 16, 12,  9, 15, 39, 16,  4, 39, 27, 30,
       26,  8])}
distribute_public_metabolite_initial:  [[300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300.
  300. 300. 300. 300. 300.]]
None
1
distribute_public_metabolite:  [[247.34445519 279.47740914 269.07808425 295.20368931 263.3524917
  286.25730487 261.73338726 271.51460974 281.59902364 274.71995052
  263.78998105 252.43323976 281.399705   279.0469058  262.31833502
  274.04429047 255.48413612 271.74579444 290.6998525 ]]
strain_number:  0 25.789473684210527
strain_various:  0 11.834734548280112
strain_number:  1 19.63157894736842
strain_various:  1 12.066528416791675
fd_r_threshold:  [1.05]
slip_r:  1.05
2
distribute_public_metabolite:  [[229.4666571  225.67359872 258.58874675 244.53291479 252.23346744
  237.74529616 243.52884072 253.70499335 240.09769725

glc__D_e_1:  -6.164092839486262
glc__D_e_1:  -14.257158921722414
glc__D_e_1:  -30.272478287874264
glc__D_e_1:  -17.339657540354374
glc__D_e_1:  -12.811628285014804
glc__D_e_1:  -8.693583291701842
glc__D_e_1:  -20.66337900098886
glc__D_e_1:  -28.119744466778407
glc__D_e_1:  -17.829885919805704
distribute_public_metabolite:  [[2.87783141 0.10758608 2.69784093 0.30919091 1.9674018  0.90124064
  2.74401303 1.28240022 0.97481549 1.21922356 2.2915577  1.13156699
  1.54764258 0.27275648 0.60981891 1.0551646  0.73619522 0.22171626
  2.49184934]]
strain_number:  0 29.157894736842106
strain_various:  0 11.065656485839595
strain_number:  1 11.578947368421053
strain_various:  1 4.258932278767139
fd_r_threshold:  [1.05, 52.60931369936053, 6.179709597684587, 7.562809138023433, 11.145916991673262, 9.095623728977216, 6.477888382312462, 3.2048804607106645, 4.509483585172561, 4.4763900344675145]
slip_r:  5.552853238328083
11
glc__D_e_1:  -15.689944443201108
glc__D_e_1:  -5.966967222202246
glc__D_e_1:  -

glc__D_e_1:  -1.1752909490429684
glc__D_e_1:  -1.6895205756746852
glc__D_e_1:  -2.159639675400237
glc__D_e_1:  -2.215231429169388
glc__D_e_1:  -2.7078705140204846
glc__D_e_1:  -2.3989816704188964
glc__D_e_1:  -0.8792031310197388
glc__D_e_1:  -4.024423292418159
glc__D_e_1:  -2.892415978583266
glc__D_e_1:  -0.4367587632339123
glc__D_e_1:  -1.2736113729907146
glc__D_e_1:  -1.1191669511899205
glc__D_e_1:  -0.9929115671920892
glc__D_e_1:  -0.06491828362961005
glc__D_e_1:  -3.763576440357493
glc__D_e_1:  -3.3002431749551113
glc__D_e_1:  -1.55381749015641
glc__D_e_1:  -0.13403168792811537
glc__D_e_1:  -1.7771091857397054
glc__D_e_1:  -1.4682203421381175
glc__D_e_1:  -0.9734280660363579
glc__D_e_1:  -1.0290198198055092
glc__D_e_1:  -3.4096896688391585
glc__D_e_1:  -2.277682355004266
glc__D_e_1:  -1.4992736099790644
glc__D_e_1:  -0.07948780775076991
glc__D_e_1:  -1.7660945537352892
glc__D_e_1:  -1.4572057101337013
glc__D_e_1:  -0.8454953555940095
glc__D_e_1:  -0.9010871093631607
glc__D_e_1:  -4

glc__D_e_1:  -0.04870790919315526
glc__D_e_1:  -0.15117250933574022
glc__D_e_1:  -0.06456738021591257
distribute_public_metabolite:  [[1.44286443 1.53708374 0.50057298 2.46368137 0.43690358 3.33824797
  1.3619397  1.34062001 0.23096874 0.11869822 2.01169063 1.85395688
  1.32590341 0.27108302 0.8587915  2.33880346 0.91101081 1.55984351
  1.87983017]]
strain_number:  0 1.0
strain_various:  0 0.7254762501100116
strain_number:  1 0.5789473684210527
strain_various:  1 0.5907880084379907
fd_r_threshold:  [1.05, 52.60931369936053, 6.179709597684587, 7.562809138023433, 11.145916991673262, 9.095623728977216, 6.477888382312462, 3.2048804607106645, 4.509483585172561, 4.4763900344675145, 4.14610992611062, 4.0321206242853265, 3.568475863503819, 4.839827429576335, 4.305420099781571, 4.007391975308642, 4.245963718820861, 4.520833333333333, 7.833333333333334, 3.8333333333333335, 5.5]
slip_r:  5.186692743764172
22
glc__D_e_1:  -0.16308363280520188
glc__D_e_1:  -0.031252292342152876
glc__D_e_1:  -0.3396

distribute_public_metabolite:  [[0.84052758 0.86433866 0.41975333 0.87803602 0.61594651 1.20690632
  0.96021065 0.61729502 0.20525557 0.75732002 1.15836167 0.63717238
  2.50523818 1.41643334 0.80004542 1.3017345  2.56083012 1.62391084
  1.48689032]]
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 52.60931369936053, 6.179709597684587, 7.562809138023433, 11.145916991673262, 9.095623728977216, 6.477888382312462, 3.2048804607106645, 4.509483585172561, 4.4763900344675145, 4.14610992611062, 4.0321206242853265, 3.568475863503819, 4.839827429576335, 4.305420099781571, 4.007391975308642, 4.245963718820861, 4.520833333333333, 7.833333333333334, 3.8333333333333335, 5.5, 4.75, 4.25, 6.5, 1.0, 0.0, 0.0, 2.0, 4.0, 0.0]
slip_r:  1.2
31
distribute_public_metabolite:  [[0.85640163 0.56265709 0.32875178 0.66498186 0.55081743 0.93291163
  0.71340088 0.40667263 3.4258584  0.9915

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

distribute_microorganism_initial:  {0: array([11, 15, 11, 11, 11, 23, 12,  1, 38,  8, 21, 17, 25, 22, 13, 35,  8,
       19,  8]), 1: array([35, 37, 32,  4, 16, 26,  8, 37, 32,  0,  6, 12, 17, 21, 21, 18, 36,
       12, 21])}
distribute_public_metabolite_initial:  [[300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300.
  300. 300. 300. 300. 300.]]
None
1
distribute_public_metabolite:  [[266.46840877 262.1638906  268.47443092 289.9155861  281.89149752
  267.30324681 286.74909739 272.04898586 250.19603291 295.06565985
  281.66031282 280.6154386  271.33772821 271.13840957 277.56454224
  263.75112897 268.27511228 279.63185356 281.02350483]]
strain_number:  0 19.157894736842106
strain_various:  0 10.989292194508621
strain_number:  1 20.894736842105264
strain_various:  1 11.937974512330964
fd_r_threshold:  [1.05]
slip_r:  1.05
2
distribute_public_metabolite:  [[226.24181805 237.93634094 242.12553195 260.86700097 255.6941848
  253.41039164 248.98399984 239.55374736 236.4837155

glc__D_e_1:  -4.966166707142366
glc__D_e_1:  -17.279790149726903
glc__D_e_1:  -18.101153281847417
glc__D_e_1:  -5.126280862798283
glc__D_e_1:  -8.188219482331478
glc__D_e_1:  -16.727065502579677
glc__D_e_1:  -14.87373244097015
glc__D_e_1:  -6.041633117125533
glc__D_e_1:  -7.628194180661281
glc__D_e_1:  -23.396464222472652
glc__D_e_1:  -12.90126489246982
glc__D_e_1:  -4.569495469629706
glc__D_e_1:  -9.106811645160345
glc__D_e_1:  -20.142792254697543
glc__D_e_1:  -11.293829865161321
glc__D_e_1:  -6.532884379420601
glc__D_e_1:  -15.988125741609394
glc__D_e_1:  -13.324356962105922
glc__D_e_1:  -7.253638826871201
glc__D_e_1:  -9.400550034156545
glc__D_e_1:  -20.386760706111936
glc__D_e_1:  -16.786899012806565
glc__D_e_1:  -6.909477370006909
glc__D_e_1:  -2.915030604593567
glc__D_e_1:  -7.88854754502087
glc__D_e_1:  -17.92184257702874
glc__D_e_1:  -9.895998657725823
glc__D_e_1:  -3.0963768297050924
glc__D_e_1:  -12.496026438124733
glc__D_e_1:  -16.43067493576004
glc__D_e_1:  -8.5592754382579

glc__D_e_1:  -3.1090466324582313
glc__D_e_1:  -1.3083652701908282
glc__D_e_1:  -2.2738397182879067
glc__D_e_1:  -1.8376389533912425
glc__D_e_1:  -3.5453007825588214
glc__D_e_1:  -3.0819675171564396
glc__D_e_1:  -0.5797524565348673
glc__D_e_1:  -5.281176687605488
glc__D_e_1:  -4.663399000402313
glc__D_e_1:  -1.9887646807281443
glc__D_e_1:  -2.0443564344972955
glc__D_e_1:  -0.6594322018493206
glc__D_e_1:  -3.84835802221108
distribute_public_metabolite:  [[3.23288557 0.80428087 3.42964343 0.69777687 2.58232032 0.48163306
  3.4783304  0.17796306 0.74943649 0.41939977 1.27386082 1.39151906
  2.07859061 1.14594608 0.30498836 0.84003335 2.11051275 0.9392286
  2.92555373]]
strain_number:  0 5.368421052631579
strain_various:  0 1.5963948303266422
strain_number:  1 4.421052631578948
strain_various:  1 1.4623625252052428
fd_r_threshold:  [1.05, 296.78750789442, 7.977230251279231, 6.788891873471228, 8.222598698608968, 7.6159583918446305, 6.750959159237445, 5.224794577605123, 3.441715717655842, 5.0

glc__D_e_1:  -0.09782928185712003
glc__D_e_1:  -0.1175156178899821
glc__D_e_1:  -0.7913327247762674
glc__D_e_1:  -0.7228075198608528
glc__D_e_1:  -0.09684691734038964
glc__D_e_1:  -0.626168013761236
glc__D_e_1:  -0.12657884849647205
glc__D_e_1:  -0.23529363045293927
distribute_public_metabolite:  [[1.39396324 1.3742769  1.01863638 0.51608504 1.00116533 0.38271044
  0.25949038 0.54937893 0.7617494  0.70045979 0.768985   2.79831644
  0.19695046 2.27679199 1.25649889 0.55371945 0.69181505 0.23505982
  0.52938728]]
strain_number:  0 1.5789473684210527
strain_various:  0 0.5907880084379908
strain_number:  1 0.6842105263157895
strain_various:  1 0.7292845505553168
fd_r_threshold:  [1.05, 296.78750789442, 7.977230251279231, 6.788891873471228, 8.222598698608968, 7.6159583918446305, 6.750959159237445, 5.224794577605123, 3.441715717655842, 5.078069047566649, 5.035408377054543, 4.0703763412915475, 5.257088273444769, 4.003483814465957, 3.707323633156967, 3.187310090702949, 3.8828401360544227, 4.16

distribute_public_metabolite:  [[1.66948961 1.21493596 0.63544695 1.08286747 0.71333376 1.52338764
  0.8754726  0.79435463 0.47934245 0.15213471 0.91521268 1.01173934
  0.93620307 0.69536156 1.53697824 1.77530157 1.9418466  2.07011837
  3.41852849]]
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 296.78750789442, 7.977230251279231, 6.788891873471228, 8.222598698608968, 7.6159583918446305, 6.750959159237445, 5.224794577605123, 3.441715717655842, 5.078069047566649, 5.035408377054543, 4.0703763412915475, 5.257088273444769, 4.003483814465957, 3.707323633156967, 3.187310090702949, 3.8828401360544227, 4.167284580498866, 4.854166666666667, 6.333333333333334, 3.2222222222222223, 6.861111111111111, 2.5, 8.0, 4.0, 3.0, 1.0, 1.0, 1.0, 1.0]
slip_r:  1.4
31
distribute_public_metabolite:  [[1.48766815 1.09223323 0.50533715 0.90075887 1.11232536 1.09979671
  0.93275506 0.72370993 0.44620634 1.57499455 0.

distribute_public_metabolite:  [[0.83826285 0.85566735 0.8608301  0.86123564 0.8645464  0.87202984
  0.87935657 0.87835493 0.85825606 0.8067656  0.71145538 0.56154936
  0.34976363 1.56552285 0.96615247 1.31438961 1.62334387 1.90911878
  3.41852849]]
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 296.78750789442, 7.977230251279231, 6.788891873471228, 8.222598698608968, 7.6159583918446305, 6.750959159237445, 5.224794577605123, 3.441715717655842, 5.078069047566649, 5.035408377054543, 4.0703763412915475, 5.257088273444769, 4.003483814465957, 3.707323633156967, 3.187310090702949, 3.8828401360544227, 4.167284580498866, 4.854166666666667, 6.333333333333334, 3.2222222222222223, 6.861111111111111, 2.5, 8.0, 4.0, 3.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
slip_r:  0.4
41
distribute_public_metabolite:  [[0.84522465 0.85355537 0.85808242 0.86129865 0.86624068 0.8726448

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

distribute_microorganism_initial:  {0: array([30, 32,  1, 12, 22, 12, 21, 25, 11, 16, 27, 22, 23,  1, 24, 22,  8,
        9, 30]), 1: array([35, 27, 20,  7, 17, 36, 18, 11, 38, 14, 24,  6, 35,  9,  9,  3, 15,
       28, 16])}
distribute_public_metabolite_initial:  [[300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300. 300.
  300. 300. 300. 300. 300.]]
None
1
distribute_public_metabolite:  [[253.12435092 260.2084401  286.13472651 287.41777144 273.81310577
  265.3079422  273.63622424 275.3497725  264.46238663 279.76988302
  265.67342483 281.1685203  258.56689855 293.49014105 278.17891312
  283.17454244 285.03554912 275.85099398 268.54743967]]
strain_number:  0 21.63157894736842
strain_various:  0 11.098651062253559
strain_number:  1 19.57894736842105
strain_various:  1 11.132294365463716
fd_r_threshold:  [1.05]
slip_r:  1.05
2
distribute_public_metabolite:  [[211.97884021 225.17672701 260.588386   258.67776743 254.41544738
  232.42868463 245.04394131 245.04613543 233.68667529

glc__D_e_1:  -8.185649653269355
glc__D_e_1:  -6.59030907403208
glc__D_e_1:  -7.5973240351108995
glc__D_e_1:  -9.239476852415798
glc__D_e_1:  -14.477171441822087
glc__D_e_1:  -10.258031030084105
glc__D_e_1:  -10.515061738277879
glc__D_e_1:  -12.704598828017746
glc__D_e_1:  -8.056997903007852
glc__D_e_1:  -5.7929832753380675
glc__D_e_1:  -4.251646493003394
glc__D_e_1:  -3.3792449632100654
glc__D_e_1:  -23.877853541361397
glc__D_e_1:  -15.234231934855304
glc__D_e_1:  -3.882279597221073
glc__D_e_1:  -3.0098780674277448
glc__D_e_1:  -19.645634260723774
glc__D_e_1:  -9.459323774322531
glc__D_e_1:  -3.8215581680405437
glc__D_e_1:  -5.408119231576292
glc__D_e_1:  -11.02924568065092
glc__D_e_1:  -5.781646015649504
glc__D_e_1:  -7.510907782093726
glc__D_e_1:  -14.562778304722595
glc__D_e_1:  -24.376858979260973
glc__D_e_1:  -11.052519033727307
glc__D_e_1:  -5.806681483032773
glc__D_e_1:  -12.311167733226675
glc__D_e_1:  -28.292009270590217
glc__D_e_1:  -10.4922917690591
glc__D_e_1:  -4.890292432

glc__D_e_1:  -1.047176046599712
glc__D_e_1:  -5.80225269578508
glc__D_e_1:  -5.184475008581904
glc__D_e_1:  -1.5307859356824634
glc__D_e_1:  -0.6027926521199842
glc__D_e_1:  -7.2896850383325535
glc__D_e_1:  -5.180114832463563
glc__D_e_1:  -0.9301748387337192
glc__D_e_1:  -1.9693516298345015
glc__D_e_1:  -3.2484490866028644
glc__D_e_1:  -1.4477677243354612
glc__D_e_1:  -2.900739974972221
glc__D_e_1:  -5.415294322070448
glc__D_e_1:  -1.7041976008432727
glc__D_e_1:  -2.063982805674195
distribute_public_metabolite:  [[1.65646492 0.636308   1.59794581 1.04113777 1.33814417 1.56631202
  0.22619607 0.51649312 0.99318435 0.20228002 1.04445454 0.44461647
  1.58943675 0.88899987 1.59379692 1.01423341 1.93918815 0.55187575
  1.32297307]]
strain_number:  0 6.2631578947368425
strain_various:  0 2.172607099005294
strain_number:  1 3.263157894736842
strain_various:  1 1.5163010832513613
fd_r_threshold:  [1.05, 510.4265613027387, 7.970181377139134, 10.069481392598417, 11.449715130673905, 10.0903096885

glc__D_e_1:  -0.2192489225668296
glc__D_e_1:  -0.012166058181455552
glc__D_e_1:  -0.9723237765260078
glc__D_e_1:  -0.9987475850995868
glc__D_e_1:  -0.1756291148662822
glc__D_e_1:  -0.815569731479374
glc__D_e_1:  -0.5856553410516511
glc__D_e_1:  -0.23927481187115407
glc__D_e_1:  -0.044202333733778465
glc__D_e_1:  -0.1283239415888846
distribute_public_metabolite:  [[0.34087601 1.54115949 1.24071014 0.25576501 1.2725436  0.53567559
  1.87991973 0.5133209  1.55242205 1.47962646 0.51946874 0.34825216
  3.21132676 0.67622279 0.90613718 2.65588855 1.44759018 2.76683942
  0.66748738]]
strain_number:  0 1.4736842105263157
strain_various:  0 0.6781104593013224
strain_number:  1 0.3157894736842105
strain_various:  1 0.46482951928041294
fd_r_threshold:  [1.05, 510.4265613027387, 7.970181377139134, 10.069481392598417, 11.449715130673905, 10.090309688592928, 10.582149453097255, 5.550381412514973, 6.5781814347213325, 4.681123335439508, 4.799893467493069, 6.486857487077613, 4.195232992391495, 4.191404

glc__D_e_1:  -0.0051000822421499725
distribute_public_metabolite:  [[1.36400411 1.27586766 1.33268543 1.4469084  1.56804808 1.33746966
  1.48669244 1.29529952 1.35916046 1.23525474 1.3814636  1.16239318
  0.18391086 1.74966524 1.75844353 1.63118988 1.674648   2.08651412
  2.88795455]]
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 510.4265613027387, 7.970181377139134, 10.069481392598417, 11.449715130673905, 10.090309688592928, 10.582149453097255, 5.550381412514973, 6.5781814347213325, 4.681123335439508, 4.799893467493069, 6.486857487077613, 4.195232992391495, 4.191404621104714, 3.7006942869740493, 4.7109553911096125, 4.514513888888889, 8.664410430839002, 4.434722222222222, 5.868055555555555, 5.611111111111111, 5.111111111111111, 2.75, 6.5, 6.0, 0.0, 4.0, 0.0, 0.0, 1.0]
slip_r:  1.0
31
distribute_public_metabolite:  [[1.35225258 1.29362802 1.34270751 1.4491669  1.52145346 1.3818972
  1.447